# 05 - DINOv2 ile Görsel Embedding Oluşturma

Bu notebook içerisinde CHRD veri setindeki gerçek görüntüler Meta AI tarafından geliştirilen DINOv2 modeli kullanılarak özellik (feature) vektörlerine dönüştürülmektedir.

Bu işlem sayesinde her görsel, içeriğini temsil eden yüksek boyutlu sayısal bir embedding'e dönüştürülür.

Bu embeddingler daha sonraki aşamalarda;

- Kültürel benzerlik analizi
- Görsel kümelenme (Clustering)
- t-SNE / PCA görselleştirmeleri
- Gerçek ve üretilmiş görüntülerin karşılaştırılması

için kullanılacaktır.

In [ ]:
import os
import torch
import torchvision.transforms as transforms
from PIL import Image
import pandas as pd
import numpy as np

print("PyTorch :", torch.__version__)

PyTorch : 2.13.0+cpu


## Veri Setini Okuma

Bu bölümde daha önce oluşturulan metadata dosyası okunmaktadır.

DINOv2 modeli bu metadata içerisindeki dosya isimlerini kullanarak ilgili görselleri işleyecektir.

In [ ]:
df = pd.read_csv(
    "../data/CHRD_Real/metadata_selected.csv",
    sep=";"
)

df.head()

,image_id,file_name,theme,location,motif_1,motif_2,season,contains_people,quality_score,cultural_value,notes
0,CHRD_REAL_0001,CHRD_REAL_0001.jpeg,Rock-cut Heritage,Ürgüp,Cave House,Stone House,Summer,No,NaN,High,NaN
1,CHRD_REAL_0004,CHRD_REAL_0004.jpeg,Village Square,Ürgüp,Street,Local People,Summer,No,NaN,Medium,NaN
2,CHRD_REAL_0005,CHRD_REAL_0005.jpeg,Fairy Chimney,NaN,Rock Formation,Balloon,Summer,No,NaN,High,NaN
3,CHRD_REAL_0006,CHRD_REAL_0006.jpeg,Local Architecture,NaN,Church,Monastery,Summer,No,NaN,Low,NaN
4,CHRD_REAL_0007,CHRD_REAL_0007.jpeg,Daily Life,NaN,Stone House,Street,Summer,No,NaN,Medium,NaN


## Görsel Klasörünü Tanımlama

Gerçek görüntüler CHRD_Real klasörü içerisinde bulunmaktadır.

In [ ]:
IMAGE_FOLDER = "../data/CHRD_Real/images"

print(os.path.exists(IMAGE_FOLDER))

False


# DINOv2 Modelinin Yüklenmesi

Bu bölümde Meta AI tarafından geliştirilen DINOv2 modeli yüklenmektedir.

DINOv2, kendi kendine denetimli (self-supervised) öğrenme yaklaşımıyla eğitilmiş güçlü bir görsel temsil modelidir.

Bu model sayesinde her görüntü, içeriğini temsil eden sayısal bir özellik vektörüne (embedding) dönüştürülecektir.

Bu embeddingler ilerleyen notebooklarda benzerlik analizi ve görselleştirme işlemlerinde kullanılacaktır.

In [ ]:
from transformers import AutoImageProcessor, AutoModel

In [ ]:
processor = AutoImageProcessor.from_pretrained("facebook/dinov2-base")

model = AutoModel.from_pretrained("facebook/dinov2-base")

model.eval()

print("DINOv2 modeli başarıyla yüklendi.")

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

DINOv2 modeli başarıyla yüklendi.


# Görsellerin DINOv2 Modeline Hazırlanması

Bu bölümde DINOv2 modeline gönderilecek görüntüler okunmaktadır.

Her görüntü:

- diskten okunur,
- RGB formatına dönüştürülür,
- DINOv2 modelinin beklediği giriş formatına hazırlanır.

Bu işlem sonucunda görüntüler model tarafından işlenebilir hale gelir.

In [ ]:
def load_image(image_path):

    image = Image.open(image_path).convert("RGB")

    inputs = processor(images=image, return_tensors="pt")

    return inputs

# Görsellerden Embedding Üretilmesi

Bu bölümde veri setindeki tüm görseller sırayla işlenmektedir.

Her görüntü için:

1. Görsel okunur.
2. DINOv2 modeline gönderilir.
3. Model tarafından oluşturulan embedding alınır.
4. Embedding bir listeye eklenir.

Notebook sonunda tüm veri seti için embedding matrisi elde edilmiş olacaktır.

In [ ]:
embeddings = []

image_names = []

In [ ]:
import pandas as pd

df = pd.read_csv(
    "../data/CHRD_Real/metadata_selected.csv",
    sep=";"
)

df.head()

,image_id,file_name,theme,location,motif_1,motif_2,season,contains_people,quality_score,cultural_value,notes
0,CHRD_REAL_0001,CHRD_REAL_0001.jpeg,Rock-cut Heritage,Ürgüp,Cave House,Stone House,Summer,No,NaN,High,NaN
1,CHRD_REAL_0004,CHRD_REAL_0004.jpeg,Village Square,Ürgüp,Street,Local People,Summer,No,NaN,Medium,NaN
2,CHRD_REAL_0005,CHRD_REAL_0005.jpeg,Fairy Chimney,NaN,Rock Formation,Balloon,Summer,No,NaN,High,NaN
3,CHRD_REAL_0006,CHRD_REAL_0006.jpeg,Local Architecture,NaN,Church,Monastery,Summer,No,NaN,Low,NaN
4,CHRD_REAL_0007,CHRD_REAL_0007.jpeg,Daily Life,NaN,Stone House,Street,Summer,No,NaN,Medium,NaN


In [ ]:
import os

IMAGE_FOLDER = "../data/CHRD_Real/raw"

print("Görsel klasörü:", IMAGE_FOLDER)
print("Klasör mevcut mu?", os.path.exists(IMAGE_FOLDER))

Görsel klasörü: ../data/CHRD_Real/raw
Klasör mevcut mu? True


In [ ]:
print(df.shape)

(164, 11)


In [ ]:
embeddings = []
image_names = []

for file_name in df["file_name"]:

    image_path = os.path.join(IMAGE_FOLDER, file_name)

    if not os.path.exists(image_path):
        continue

    inputs = load_image(image_path)

    with torch.no_grad():
        outputs = model(**inputs)

    embedding = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()

    embeddings.append(embedding)
    image_names.append(file_name)

# Embedding Boyutunun Kontrol Edilmesi

Bu bölümde oluşturulan embeddinglerin sayısı ve boyutu kontrol edilmektedir.

Her satır bir görüntüyü,

Her sütun ise DINOv2 tarafından öğrenilen bir özelliği temsil etmektedir.

In [ ]:
import numpy as np

In [ ]:
embeddings = np.array(embeddings)

print("Embedding Shape :", embeddings.shape)

Embedding Shape : (82, 768)


# Embeddinglerin Kaydedilmesi

Üretilen embeddingler sonraki notebooklarda tekrar kullanılacağından diske kaydedilmektedir.

Böylece aynı işlem her çalıştırmada yeniden yapılmak zorunda kalmayacaktır.

In [ ]:
os.makedirs("../embeddings", exist_ok=True)

In [ ]:
np.save("../embeddings/chrd_real_embeddings.npy", embeddings)

In [ ]:
pd.DataFrame({
    "file_name": image_names
}).to_csv(
    "../embeddings/chrd_real_image_names.csv",
    index=False
)

In [ ]:
print("Embedding dosyaları başarıyla kaydedildi.")

print()

print("Toplam Görsel :", len(image_names))

print("Embedding Boyutu :", embeddings.shape)

Embedding dosyaları başarıyla kaydedildi.

Toplam Görsel : 82
Embedding Boyutu : (82, 768)


In [ ]:
print("IMAGE_FOLDER:")
print(IMAGE_FOLDER)

print()

print("İlk 10 dosya:")

for file in df["file_name"].head(10):
    print(file)

print()

print("Kontrol:")

for file in df["file_name"].head(10):

    path = os.path.join(IMAGE_FOLDER, file)

    print(path, "---->", os.path.exists(path))

IMAGE_FOLDER:
../data/CHRD_Real/raw

İlk 10 dosya:
CHRD_REAL_0001.jpeg
CHRD_REAL_0004.jpeg
CHRD_REAL_0005.jpeg
CHRD_REAL_0006.jpeg
CHRD_REAL_0007.jpeg
CHRD_REAL_0008.jpeg
CHRD_REAL_0009.jpeg
CHRD_REAL_0011.jpeg
CHRD_REAL_0014.jpeg
CHRD_REAL_0015.jpeg

Kontrol:
../data/CHRD_Real/raw\CHRD_REAL_0001.jpeg ----> True
../data/CHRD_Real/raw\CHRD_REAL_0004.jpeg ----> True
../data/CHRD_Real/raw\CHRD_REAL_0005.jpeg ----> True
../data/CHRD_Real/raw\CHRD_REAL_0006.jpeg ----> True
../data/CHRD_Real/raw\CHRD_REAL_0007.jpeg ----> True
../data/CHRD_Real/raw\CHRD_REAL_0008.jpeg ----> True
../data/CHRD_Real/raw\CHRD_REAL_0009.jpeg ----> True
../data/CHRD_Real/raw\CHRD_REAL_0011.jpeg ----> True
../data/CHRD_Real/raw\CHRD_REAL_0014.jpeg ----> True
../data/CHRD_Real/raw\CHRD_REAL_0015.jpeg ----> True


In [ ]:
import os

print(os.listdir("../data/CHRD_Real/raw")[:20])

['CHRD_REAL_0001.jpeg', 'CHRD_REAL_0002.jpeg', 'CHRD_REAL_0003.jpeg', 'CHRD_REAL_0004.jpeg', 'CHRD_REAL_0005.jpeg', 'CHRD_REAL_0006.jpeg', 'CHRD_REAL_0007.jpeg', 'CHRD_REAL_0008.jpeg', 'CHRD_REAL_0009.jpeg', 'CHRD_REAL_0010.jpeg', 'CHRD_REAL_0011.jpeg', 'CHRD_REAL_0012.jpeg', 'CHRD_REAL_0013.jpeg', 'CHRD_REAL_0014.jpeg', 'CHRD_REAL_0015.jpeg', 'CHRD_REAL_0016.jpeg', 'CHRD_REAL_0017.jpeg', 'CHRD_REAL_0018.jpeg', 'CHRD_REAL_0019.jpeg', 'CHRD_REAL_0020.jpeg']
